In [2]:
import pymupdf
from pathlib import Path

PDF_PATH = Path("data/try2.pdf")

# Extract text page by page
pages = []

with pymupdf.open(PDF_PATH) as pdf:
    for page_num, page in enumerate(pdf):
        text = page.get_text("text").strip()

        if text:
            pages.append({
                "page": page_num + 1,
                "text": text
            })

print("Pages with text:", len(pages))
print("\nSample text:\n", pages[0]["text"][:500])

Pages with text: 325

Sample text:
 Muncie, Indiana 
 
 
FACULTY 
AND 
PROFESSIONAL 
PERSONNEL HANDBOOK 
 
ACADEMIC YEAR: 2026-2027 
 
 
 
 
Compiled by the Office of the Provost 
and Executive Vice President for Academic Affairs


In [3]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks = []

for page in pages:
    text = page["text"]
    start = 0

    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "chunk_id": len(chunks),
                "page": page["page"],
                "text": chunk_text
            })

        if end == len(text):
            break

        start = end - CHUNK_OVERLAP

print("Total chunks:", len(chunks))
print("\nSample chunk:\n", chunks[0]["text"])

Total chunks: 1257

Sample chunk:
 Muncie, Indiana 
 
 
FACULTY 
AND 
PROFESSIONAL 
PERSONNEL HANDBOOK 
 
ACADEMIC YEAR: 2026-2027 
 
 
 
 
Compiled by the Office of the Provost 
and Executive Vice President for Academic Affairs


## Building the baseline vector index

In [4]:

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load the embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Embed all chunks
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(embeddings, dtype="float32")

# Build FAISS cosine-similarity index
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Chunks indexed:", index.ntotal)
print("Embedding dimensions:", embeddings.shape[1])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Chunks indexed: 1257
Embedding dimensions: 384


## Test Retrieval

In [5]:
def retrieve_baseline(query, top_k=10):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    results = []
    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]), start=1
    ):
        results.append({
            "rank": rank,
            "score": float(score),
            **chunks[int(idx)]
        })

    return results


query = "What is the policy for taking leave from work?"

results = retrieve_baseline(query)

for result in results[:3]:
    print(
        f"\nRank {result['rank']} | "
        f"Score: {result['score']:.3f} | "
        f"Page: {result['page']}"
    )
    print(result["text"][:400])



Rank 1 | Score: 0.539 | Page: 199
employed elsewhere. 
May be subject to tax 
if employed elsewhere. 
 
NOTES: 
A. In the event an employee on leave fails to return as agreed, the employee must repay proportional leave 
payments. Leave payments include: (1) salary, (2) the University’s contributions to retirement plans made on 
behalf of the employee, and (3) the University’s portion paid for benefit programs during the period of 

Rank 2 | Score: 0.523 | Page: 194
ear. 
 
48.6.5. 
Parental Leave Policy 
48.6.5.1. 
This policy takes effect on July 1, 2017 and applies to births or adoptions that occur on 
July 1, 2017 or after that date. This policy eliminates the former Pregnancy, Maternity, 
and Child Care Leave policies. 
48.6.5.2. 
Ball State University offers up to twenty-six (26) weeks of leave to eligible full and part 
time faculty, professional, and 

Rank 3 | Score: 0.488 | Page: 192
ntly allowed for routine care. 
 
48.4. MILITARY LEAVE 71  
48.4.1. 
A leave of absence will 

### More Precise Question

In [6]:
query = "How many weeks of parental leave are eligible faculty members entitled to?"
results = retrieve_baseline(query)

for result in results[:5]:
    print(
        f"\nRank {result['rank']} | "
        f"Score: {result['score']:.3f} | "
        f"Page: {result['page']}"
    )
    print(result["text"][:500])


Rank 1 | Score: 0.800 | Page: 194
Leaves. 
 
48.6.5.3. 
Eligibility for Parental Leave 
48.6.5.3.1. 
This benefit is available to full and part-time faculty, professional and staff 
employees. Temporary employees, semester non-tenure-line faculty, service 
employees, graduate assistants and student employees are not covered by 
this policy. 
48.6.5.3.2. 
To be eligible for paid parental leave, an employee must be appointed to a 
full-time, benefit eligible position and been employed by the University for 
twelve (12) continuous 

Rank 2 | Score: 0.755 | Page: 195
re-track faculty members who are eligible for parental leave (paid or 
unpaid) may request an extension of their probationary period as outlined 
in the University Promotion and Tenure Document. 
 
48.6.5.5. 
Paid Parental Leave General Provisions 
48.6.5.5.1. 
Eligible employees are entitled to a maximum of six (6) weeks of paid 
parental leave following the birth or adoption of a child. Paid parental leave 
does not reduce 

In [7]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
from IPython.display import display

query = "How many weeks of parental leave are eligible faculty members entitled to?"

# Retrieve the top 10 candidates
results = retrieve_baseline(query, top_k=10)

# Create a report-friendly table
baseline_table = pd.DataFrame([
    {
        "Query": query,
        "Original Rank": r["rank"],
        "Similarity Score": round(r["score"], 4),
        "Chunk ID": r["chunk_id"],
        "PDF Page": r["page"],
        "Retrieved Text": r["text"]
    }
    for r in results
])

display(baseline_table)


,Query,Original Rank,Similarity Score,Chunk ID,PDF Page,Retrieved Text
0,How many weeks of parental leave are eligible ...,1,0.8001,736,194,Leaves. \n \n48.6.5.3. \nEligibility for Paren...
1,How many weeks of parental leave are eligible ...,2,0.7551,740,195,re-track faculty members who are eligible for ...
2,How many weeks of parental leave are eligible ...,3,0.7291,738,195,irths or adoptions do not increase the amount ...
3,How many weeks of parental leave are eligible ...,4,0.6932,735,194,ear. \n \n48.6.5. \nParental Leave Policy \n48...
4,How many weeks of parental leave are eligible ...,5,0.6664,744,196,qual to the \nlength of the leave that was not...
5,How many weeks of parental leave are eligible ...,6,0.6612,737,195,Ball State University |2026-2027 Faculty and P...
6,How many weeks of parental leave are eligible ...,7,0.6492,739,195,sociate Vice President. Faculty who elect to \...
7,How many weeks of parental leave are eligible ...,8,0.6351,741,196,Ball State University |2026-2027 Faculty and P...
8,How many weeks of parental leave are eligible ...,9,0.6053,716,190,s made to the Board of \nTrustees and will ann...
9,How many weeks of parental leave are eligible ...,10,0.6052,743,196,f sick \nleave for that event. A birth mother ...


In [9]:
baseline_table.to_csv(
    "baseline_parental_leave_results.csv",
    index=False
)

print("Saved baseline results.")

Saved baseline results.


## Cross-Encoder

In [10]:
from sentence_transformers import CrossEncoder

# Load the re-ranking model
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Score each query-chunk pair
pairs = [
    (query, result["text"])
    for result in results
]

rerank_scores = reranker.predict(pairs)

# Attach scores and sort from highest to lowest
reranked_results = []

for result, score in zip(results, rerank_scores):
    item = result.copy()
    item["rerank_score"] = float(score)
    reranked_results.append(item)

reranked_results.sort(
    key=lambda x: x["rerank_score"],
    reverse=True
)

# Display the new ranking
for rank, result in enumerate(reranked_results, start=1):
    print(
        f"\nNew Rank {rank} | "
        f"Original Rank {result['rank']} | "
        f"Score: {result['rerank_score']:.3f} | "
        f"Page: {result['page']}"
    )
    print(result["text"][:350])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


New Rank 1 | Original Rank 2 | Score: 8.364 | Page: 195
re-track faculty members who are eligible for parental leave (paid or 
unpaid) may request an extension of their probationary period as outlined 
in the University Promotion and Tenure Document. 
 
48.6.5.5. 
Paid Parental Leave General Provisions 
48.6.5.5.1. 
Eligible employees are entitled to a maximum of six (6) weeks of paid 
parental leave fo

New Rank 2 | Original Rank 1 | Score: 5.949 | Page: 194
Leaves. 
 
48.6.5.3. 
Eligibility for Parental Leave 
48.6.5.3.1. 
This benefit is available to full and part-time faculty, professional and staff 
employees. Temporary employees, semester non-tenure-line faculty, service 
employees, graduate assistants and student employees are not covered by 
this policy. 
48.6.5.3.2. 
To be eligible for paid par

New Rank 3 | Original Rank 4 | Score: 4.893 | Page: 194
ear. 
 
48.6.5. 
Parental Leave Policy 
48.6.5.1. 
This policy takes effect on July 1, 2017 and applies to births or adoptions 

In [11]:
reranked_table = pd.DataFrame([
    {
        "Query": query,
        "Original Rank": r["rank"],
        "New Rank": new_rank,
        "Cross-Encoder Score": round(r["rerank_score"], 4),
        "Chunk ID": r["chunk_id"],
        "PDF Page": r["page"],
        "Retrieved Text": r["text"]
    }
    for new_rank, r in enumerate(reranked_results, start=1)
])

display(reranked_table)

reranked_table.to_csv(
    "reranked_parental_leave_results.csv",
    index=False
)

,Query,Original Rank,New Rank,Cross-Encoder Score,Chunk ID,PDF Page,Retrieved Text
0,How many weeks of parental leave are eligible ...,2,1,8.3640,740,195,re-track faculty members who are eligible for ...
1,How many weeks of parental leave are eligible ...,1,2,5.9486,736,194,Leaves. \n \n48.6.5.3. \nEligibility for Paren...
2,How many weeks of parental leave are eligible ...,4,3,4.8935,735,194,ear. \n \n48.6.5. \nParental Leave Policy \n48...
3,How many weeks of parental leave are eligible ...,6,4,3.9593,737,195,Ball State University |2026-2027 Faculty and P...
4,How many weeks of parental leave are eligible ...,3,5,2.8156,738,195,irths or adoptions do not increase the amount ...
5,How many weeks of parental leave are eligible ...,7,6,2.1652,739,195,sociate Vice President. Faculty who elect to \...
6,How many weeks of parental leave are eligible ...,8,7,1.5659,741,196,Ball State University |2026-2027 Faculty and P...
7,How many weeks of parental leave are eligible ...,5,8,0.7786,744,196,qual to the \nlength of the leave that was not...
8,How many weeks of parental leave are eligible ...,10,9,-0.9322,743,196,f sick \nleave for that event. A birth mother ...
9,How many weeks of parental leave are eligible ...,9,10,-4.9957,716,190,s made to the Board of \nTrustees and will ann...


## Test Gemini Connection

In [12]:

import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "API key not found. Check that .env is in your project root."
    )

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")


Gemini client initialized successfully!


## Generate Answer 1 Vs Answer 2

In [15]:
import time

# Use the same model and prompt for both answers
MODEL = "gemini-3.5-flash"

def generate_answer(query, retrieved_chunks):
    context = "\n\n".join(
        f"[Page {r['page']} | Chunk {r['chunk_id']}]\n{r['text']}"
        for r in retrieved_chunks
    )

    prompt = f"""
You are answering questions using the provided university handbook.

Answer the question using ONLY the supplied context.
Be concise, accurate, and explicit about important conditions.
If the context does not contain the answer, say so.
Cite the relevant page number(s).

Context:
{context}

Question: {query}

Answer:
"""

    start = time.perf_counter()

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    elapsed = time.perf_counter() - start
    return response.text, elapsed


# Use the same query for both pipelines
query = "How many weeks of parental leave are eligible faculty members entitled to?"

# Pipeline 1: original baseline top 3
baseline_top3 = results[:3]

# Pipeline 2: re-ranked top 3
reranked_top3 = reranked_results[:3]

answer1, time1 = generate_answer(query, baseline_top3)
answer2, time2 = generate_answer(query, reranked_top3)

print("===== ANSWER 1: BASELINE =====")
print(answer1)
print(f"\nGeneration time: {time1:.2f} seconds")

print("\n===== ANSWER 2: RE-RANKED =====")
print(answer2)
print(f"\nGeneration time: {time2:.2f} seconds")


===== ANSWER 1: BASELINE =====
Based on the provided context, eligible faculty members are entitled to:

* **Paid Parental Leave:** A maximum of **six (6) weeks** of paid parental leave following the birth or adoption of a child (Page 195).
* **Total Parental Leave (Paid/Unpaid):** Up to **twenty-six (26) weeks** of parental leave in total (Page 195).

Generation time: 4.90 seconds

===== ANSWER 2: RE-RANKED =====
Based on the provided context, eligible faculty members are entitled to:

* **Up to twenty-six (26) weeks** of total parental leave (paid and unpaid combined) for the birth or adoption of a child (Page 194).
* **A maximum of six (6) weeks** of *paid* parental leave, paid at 100% of regular pay (Page 195).

**Important Conditions:**
* **General Parental Leave Eligibility:** Available to eligible full- and part-time faculty. Temporary employees and semester non-tenure-line faculty are not covered (Page 194).
* **Paid Parental Leave Eligibility:** To receive paid parental leave,

## Stress Tests

In [20]:
import pandas as pd
import time

stress_queries = [
    "How many weeks of parental leave are eligible faculty members entitled to?",
    "Does parental leave apply when an employee's new spouse adopts a child?",
    "How far in advance should parental leave be requested?",
    "Which categories of employees are excluded from parental leave?"
]

stress_results = []
timing_results = []

for q in stress_queries:

    # --------------------------------------------------
    # Stage 1: Retrieve Top 10 (shared by both pipelines)
    # --------------------------------------------------
    start = time.perf_counter()
    baseline = retrieve_baseline(q, top_k=10)
    retrieval_time = time.perf_counter() - start

    # --------------------------------------------------
    # Baseline pipeline: Retrieval + Top 3 + LLM answer
    # --------------------------------------------------
    baseline_top3 = baseline[:3]

    start = time.perf_counter()
    baseline_answer, _ = generate_answer(q, baseline_top3)
    baseline_generation_time = time.perf_counter() - start

    baseline_total = retrieval_time + baseline_generation_time

    # --------------------------------------------------
    # Reranked pipeline: Retrieval + Reranking + LLM
    # --------------------------------------------------
    start = time.perf_counter()

    pairs = [(q, item["text"]) for item in baseline]
    scores = reranker.predict(pairs)

    ranked = []
    for item, score in zip(baseline, scores):
        new_item = item.copy()
        new_item["rerank_score"] = float(score)
        ranked.append(new_item)

    ranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    reranking_time = time.perf_counter() - start

    reranked_top3 = ranked[:3]

    start = time.perf_counter()
    reranked_answer, _ = generate_answer(q, reranked_top3)
    reranked_generation_time = time.perf_counter() - start

    reranked_total = (
        retrieval_time
        + reranking_time
        + reranked_generation_time
    )

    # --------------------------------------------------
    # Save pipeline timings
    # --------------------------------------------------
    timing_results.append({
        "Query": q,
        "Retrieval (s)": round(retrieval_time, 4),
        "Reranking (s)": round(reranking_time, 4),
        "Baseline Generation (s)": round(baseline_generation_time, 4),
        "Reranked Generation (s)": round(reranked_generation_time, 4),
        "Baseline Total (s)": round(baseline_total, 4),
        "Reranked Total (s)": round(reranked_total, 4),
        "Baseline Answer": baseline_answer,
        "Reranked Answer": reranked_answer
    })

    # --------------------------------------------------
    # Save original and reranked chunk ranks
    # --------------------------------------------------
    for new_rank, item in enumerate(ranked, start=1):
        stress_results.append({
            "Query": q,
            "Original Rank": item["rank"],
            "New Rank": new_rank,
            "Rank Change": item["rank"] - new_rank,
            "Page": item["page"],
            "Cross-Encoder Score": round(
                item["rerank_score"], 3
            ),
            "Text": item["text"][:350]
        })


# ------------------------------------------------------
# Display and save ranking results
# ------------------------------------------------------
stress_df = pd.DataFrame(stress_results)

display(
    stress_df.sort_values(
        ["Query", "New Rank"]
    ).reset_index(drop=True)
)

stress_df.to_csv(
    "stress_test_rankings.csv",
    index=False
)

# ------------------------------------------------------
# Display and save latency results
# ------------------------------------------------------
timing_df = pd.DataFrame(timing_results)

display(
    timing_df[[
        "Query",
        "Retrieval (s)",
        "Reranking (s)",
        "Baseline Generation (s)",
        "Reranked Generation (s)",
        "Baseline Total (s)",
        "Reranked Total (s)"
    ]]
)

print("\nAverage timings:")
display(
    timing_df[[
        "Retrieval (s)",
        "Reranking (s)",
        "Baseline Generation (s)",
        "Reranked Generation (s)",
        "Baseline Total (s)",
        "Reranked Total (s)"
    ]].mean().round(4).to_frame("Average (seconds)")
)

timing_df.to_csv(
    "stress_test_latency.csv",
    index=False
)

,Query,Original Rank,New Rank,Rank Change,Page,Cross-Encoder Score,Text
0,Does parental leave apply when an employee's n...,6,1,5,195,4.351,Ball State University |2026-2027 Faculty and P...
1,Does parental leave apply when an employee's n...,1,2,-1,194,3.353,ear. \n \n48.6.5. \nParental Leave Policy \n48...
2,Does parental leave apply when an employee's n...,2,3,-1,195,2.050,re-track faculty members who are eligible for ...
3,Does parental leave apply when an employee's n...,7,4,3,192,1.343,ave. The employee must make arrangements with ...
4,Does parental leave apply when an employee's n...,3,5,-2,194,0.720,Leaves. \n \n48.6.5.3. \nEligibility for Paren...
5,Does parental leave apply when an employee's n...,4,6,-2,196,0.454,ave not used in \naccordance with this policy ...
6,Does parental leave apply when an employee's n...,5,7,-2,195,0.164,irths or adoptions do not increase the amount ...
7,Does parental leave apply when an employee's n...,8,8,0,196,-3.802,f sick \nleave for that event. A birth mother ...
8,Does parental leave apply when an employee's n...,9,9,0,196,-6.798,qual to the \nlength of the leave that was not...
9,Does parental leave apply when an employee's n...,10,10,0,192,-6.871,ders are \nterminated. An employee taking such...


,Query,Retrieval (s),Reranking (s),Baseline Generation (s),Reranked Generation (s),Baseline Total (s),Reranked Total (s)
0,How many weeks of parental leave are eligible ...,0.0480,0.5979,5.2854,3.5483,5.3333,4.1942
1,Does parental leave apply when an employee's n...,0.0273,1.2700,4.1668,3.5656,4.1941,4.8629
2,How far in advance should parental leave be re...,0.0356,1.3198,3.8010,2.8619,3.8366,4.2173
3,Which categories of employees are excluded fro...,0.0476,1.0507,2.9235,3.0460,2.9710,4.1443



Average timings:


,Average (seconds)
Retrieval (s),0.0396
Reranking (s),1.0596
Baseline Generation (s),4.0442
Reranked Generation (s),3.2554
Baseline Total (s),4.0838
Reranked Total (s),4.3547


## Generate Answers for the Stress Tests

In [19]:

stress_answer_results = []

for q in stress_queries:
    # Retrieve the same top 10 candidates
    baseline = retrieve_baseline(q, top_k=10)

    # Re-rank those candidates
    scores = reranker.predict(
        [(q, item["text"]) for item in baseline]
    )

    ranked = []
    for item, score in zip(baseline, scores):
        entry = item.copy()
        entry["rerank_score"] = float(score)
        ranked.append(entry)

    ranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # Generate answers using the top 3 from each pipeline
    baseline_answer, baseline_time = generate_answer(
        q, baseline[:3]
    )
    reranked_answer, reranked_time = generate_answer(
        q, ranked[:3]
    )

    stress_answer_results.append({
        "query": q,
        "baseline_answer": baseline_answer,
        "reranked_answer": reranked_answer,
        "baseline_time": baseline_time,
        "reranked_time": reranked_time,
        "baseline_top3": baseline[:3],
        "reranked_top3": ranked[:3]
    })

    print("\n" + "=" * 80)
    print("QUERY:", q)
    print("\nBASELINE ANSWER:\n", baseline_answer)
    print("\nRERANKED ANSWER:\n", reranked_answer)
    print(
        f"\nGeneration times: baseline={baseline_time:.2f}s, "
        f"reranked={reranked_time:.2f}s"
    )



QUERY: How many weeks of parental leave are eligible faculty members entitled to?

BASELINE ANSWER:
 Based on the provided handbook, eligible faculty members are entitled to:

* **Paid Parental Leave:** A maximum of **six (6) weeks** of paid parental leave (paid at 100% of regular pay) following the birth or adoption of a child (Page 195). 
* **Total Parental Leave (Paid/Unpaid):** Up to **twenty-six (26) weeks** of combined paid and unpaid parental leave (Page 195). 

*(Page 195)*

RERANKED ANSWER:
 Based on the provided context:

* **Total Parental Leave (paid and unpaid):** Eligible full- and part-time faculty members are offered up to **twenty-six (26) weeks** of parental leave for the birth or adoption of a child (Page 194).
* **Paid Parental Leave:** Within that leave, eligible employees (those appointed to a full-time, benefit-eligible position with 12 continuous months of service) are entitled to a maximum of **six (6) weeks** of paid parental leave at 100% regular pay (Page 1